# 1. Libraries

In [ ]:
import time
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc
import scvi
from IPython.display import display

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150)
sc.logging.print_header()

%matplotlib inline

# 2. Variables

In [ ]:
# Path to the object to be integrated
adata = sc.read_h5ad("/inputdirectory/yourdata.h5ad")

# Paths for the saved outputs
output_file = "/outputdirectory/yourdata_integrated.h5ad"
model_dir   = "/outputdirectory/yourdata_integration"

# Number of k_neighbors for integration
K_NEIGHBORS = 30

# Leiden resolutions to be investigated
resolutions = [0.1, 3.0]

# Species of the dataset; selects the marker gene list in section 7
SPECIES = 'mouse'        # 'mouse' or 'human'

# Resolution used to inspect and remove clusters (sections 8 to 12)
CLEAN_RES = 3.0          # must be in `resolutions`; write as a float (1.0, not 1)

# Putative low quality clusters to be investigated further;
    # identified after sections 6 & 7, and used for 8 to 11
low = ['0', '1', '2', '3', '4', '5', '6']

# Clusters to be removed;
    # identified after section 11, and used in section 12
remove = ['1', '4']

# Clusters to be annotated;
# Cell type annotation; identified after section 11, used for section 13
TYPE_RES = 0.1           # the Leiden resolution annotation will be performed; 
                         # must be in `resolutions`
                         # write as a float (1.0, not 1)
type_map = {
    'epi': 'all',        # 'all' = every cell, regardless of cluster
}

# Cell subtype annotation; identified after section 11, used for section 14
SUBTYPE_RES = 0.1        # the Leiden resolution annotation will be performed; 
                         # must be in `resolutions`
                         # write as a float (1.0, not 1)
subtype_map = {
    'epi_luminal':   ['0'],
    'epi_glandular': ['1'],
}
# Sanity check: every resolution used later must be computed in section 4
for name, r in [('CLEAN_RES', CLEAN_RES), ('TYPE_RES', TYPE_RES), ('SUBTYPE_RES', SUBTYPE_RES)]:
    assert r in resolutions, f"{name}={r} is not in resolutions {resolutions}"
assert SPECIES in ('mouse', 'human'), f"SPECIES must be 'mouse' or 'human', not '{SPECIES}'"

## 2.1. Variables used in this study for the X-Mens dataset

## 2.2. Variables used in this study for the human dataset

# 3. Prep the Data for Integration and find Highly Variable Genes

In [ ]:
# slice out mitochondrial, ribosomal genes for the sake of HVG selection
adata_hvg = adata[:, (~adata.var['mt']) & (~adata.var['ribo'])].copy()
print(f'Retained {adata_hvg.shape[0]} cells, {adata_hvg.shape[1]} genes')
print(adata_hvg)

In [ ]:
# drop genes detected in fewer than 10 nuclei; only affects the HVG candidates, adata keeps all genes)
n_det = np.asarray((adata_hvg.layers['raw_counts'] > 0).sum(axis=0)).ravel()
adata_hvg = adata_hvg[:, n_det >= 10].copy()
print(f'{adata_hvg.n_vars} genes kept for HVG selection')

In [ ]:
# select HVGs (creates adata_hvg.uns['hvg'] and keeps only the 3,000 genes)
sc.pp.highly_variable_genes(adata_hvg, flavor='seurat_v3', layer='raw_counts',
                            batch_key='sample', n_top_genes=3000, span=0.5, subset=True)
print(adata_hvg)

In [ ]:
# add back highly-variable genes to the larger adata object
adata.uns['hvg'] = adata_hvg.uns['hvg'].copy()
adata.var['highly_variable'] = adata.var.index.isin(list(adata_hvg.var[adata_hvg.var['highly_variable']].index))
adata.var['highly_variable_rank'] = adata.var.index.map(dict([*zip(adata_hvg.var.index, adata_hvg.var['highly_variable_rank'])]))
adata.var['highly_variable_nbatches'] = adata.var.index.map(dict([*zip(adata_hvg.var.index, adata_hvg.var['highly_variable_nbatches'])]))

# 4. scVI Integration

In [ ]:
scvi.settings.seed = 0   # reproducible latent space

In [ ]:
scvi.model.SCVI.setup_anndata(adata_hvg, layer="raw_counts", batch_key="sample") # note we go back to the hvg object

In [ ]:
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
model.train(early_stopping = True)

In [ ]:
SCVI_LATENT_KEY = "X_scVI"
adata.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

In [ ]:
n_neighbors = K_NEIGHBORS
SCVI_NEIGHBORS_KEY = f"neighbors_scVI_{n_neighbors}"
print(SCVI_NEIGHBORS_KEY)

In [ ]:
neighbors = [K_NEIGHBORS]
SEED = 0
for n_neighbors in neighbors:
    print("***********************")
    SCVI_NEIGHBORS_KEY = f"neighbors_scVI_{n_neighbors}"
    print(f"Computing neighbors with n_neighbors={n_neighbors}")
    
    sc.pp.neighbors(adata,
                    use_rep=SCVI_LATENT_KEY,
                    n_neighbors=n_neighbors,
                    key_added=SCVI_NEIGHBORS_KEY)
    
    for res in resolutions:
        LEIDEN_SCVI_KEY = f"leiden_scVI_{res}"
        start_time = time.time()
        print(f"Leiden clustering at resolution {res} using {SCVI_NEIGHBORS_KEY}")
        
        sc.tl.leiden(
            adata,   
            resolution=res,
            random_state=SEED,
            flavor="igraph",
            neighbors_key=SCVI_NEIGHBORS_KEY,
            key_added=LEIDEN_SCVI_KEY
        )

        end_time = time.time()
        print(f"Execution time: {end_time - start_time:.4f} seconds")
        print("-----------------------")

In [ ]:
sc.tl.umap(adata, neighbors_key = SCVI_NEIGHBORS_KEY)

In [ ]:
SCVI_UMAP_KEY = 'X_umap_scVI'
adata.obsm[SCVI_UMAP_KEY] = adata.obsm['X_umap'].copy()
del adata.obsm['X_umap']

# 5. Make a Layer for logarithmized counts

In [ ]:
# normalized + log-transformed copy
adata.layers['log1p_counts'] = adata.layers['raw_counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer='log1p_counts')
sc.pp.log1p(adata, layer='log1p_counts')

# keep raw counts in X for now
adata.X = adata.layers['raw_counts'].copy()
print(adata)

# 6. Visualize Leiden clusters on a UMAP

In [ ]:
# scVI UMAP coordinates
umap_coords = adata.obsm[SCVI_UMAP_KEY]

# up to 60 distinct colors (tab20 + tab20b + tab20c)
tab_colors = plt.cm.tab20.colors + plt.cm.tab20b.colors + plt.cm.tab20c.colors

for res in resolutions:
    clustering_key = f'leiden_scVI_{res}'
    if clustering_key not in adata.obs:
        print(f"Warning: clustering for resolution {res} not found!")
        continue

    clusters = adata.obs[clustering_key].astype(str)
    unique_clusters = sorted(clusters.unique(), key=int)
    cluster_colors = {c: tab_colors[i % len(tab_colors)] for i, c in enumerate(unique_clusters)}
    if len(unique_clusters) > len(tab_colors):
        print(f"Note: {len(unique_clusters)} clusters at resolution {res}; colors repeat")

    with plt.style.context('default'):                               # white background, black text
        fig, ax = plt.subplots(figsize=(15, 15))
        for c in unique_clusters:
            mask = (clusters == c).values
            ax.scatter(umap_coords[mask, 0], umap_coords[mask, 1],
                       s=2, color=cluster_colors[c], alpha=0.6,
                       label=f"cluster_{c}")                          
        ax.axis('off')
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', title="Clusters",
                  markerscale=5, frameon=False)
        ax.set_title(f"Leiden {res}", fontsize=16)
        fig.patch.set_facecolor('white')
        plt.tight_layout()
        plt.show()

# 7. Make Dotplots for cell type/subtype marker genes

In [ ]:
marker_dicts = {
    'mouse': {
        'Fibroblast':       ['Pdgfra', 'Col3a1', 'Col1a2', 'Celf2', 'Cdh11', 'Hand2'],
        'Decidual':         ['Wnt4', 'Cebpb', 'Prl8a2', 'Bmp2', 'Cdh3', 'Gja1', 'Gpx3', 'Ptgs2'],
        'Hypoxia':          ['Hif1a', 'Epas1', 'Vegfa'],
        'Epithelial':       ['Cdh1', 'Klf5', 'Foxa2', 'Tacstd2', 'Sox9'],
        'Endothelial':      ['Flt1'],
        'Lymphatic':        ['Lyve1'],
        'Mural / SMC':      ['Rgs5', 'Cnn1', 'Acta2'],
        'Pan-immune':       ['Ptprc'],
        'Macrophage':       ['Mrc1', 'Adgre1', 'Csf1r'],
        'Monocyte':         ['Fgr', 'Sirpb1b', 'Itgam'],
        'Neutrophil':       ['S100a9', 'Cxcr2'],
        'DC':               ['Clec9a', 'Flt3', 'Zbtb46', 'Xcr1'],
        'T cell':           ['Cd3e', 'Cd3g', 'Trbc2', 'Tcf7', 'Themis', 'Bcl11b'],
        'γδ T cell':        ['Il23r', 'Cd163l1'],
        'B cell':           ['Pax5', 'Ebf1', 'Ighm'],
        'uNK':              ['Ncr1', 'Klrk1', 'Styk1'],
        'Cytotoxic uNK':    ['Gzmb', 'Gzmg', 'Prf1'],
        'Cycling':          ['Top2a', 'Kif15', 'Knl1', 'Mki67', 'Bub1', 'Depdc1a', 'Kif11'],
        'Apoptosis':        ['Casp3', 'Casp9', 'Trp53', 'Bax'],
        'Hormone':          ['Pgr', 'Esr1'],
    },
    'human': {
        'Epithelial':       ['EPCAM', 'TACSTD2', 'UCA1', 'SDC4', 'KLF5', 'WFDC2'],
        'Ciliated':         ['FOXJ1', 'DYNLRB2', 'SNTN', 'C9orf24', 'SPMIP6', 'CDHR3', 'DYDC2'],
        'Stromal fibro':    ['COL5A1', 'COL6A1', 'COL6A3', 'LUM', 'DCN', 'CRISPLD2', 'FOXL2', 'MMP11'],
        'Decidual':         ['IGFBP1', 'PRL', 'FOXO1', 'CEBPB', 'LEFTY2', 'MMP10', 'IGF1', 'DKK1', 'IL15', 'CFD', 'STAU1'],
        'Endothelial':      ['PECAM1', 'VWF', 'ADGRL4', 'PCDH17', 'RNASE1'],
        'Perivascular':     ['RGS5', 'GUCY1A2', 'BGN', 'MCAM', 'PDGFRB'],
        'VSMC':             ['NOTCH3', 'ELN', 'PLN', 'SOST', 'JAG1'],
        'Smooth muscle':    ['MYH11', 'ACTA2', 'CNN1', 'TAGLN', 'MYOCD', 'LMOD1', 'SMTN', 'SYNPO2', 'KCNMB1'],
        'Myometrium':       ['DES', 'ACTG2', 'OXTR', 'PTGFR'],
        'Immune (all)':     ['PTPRC', 'STK17B'],
        'Myeloid':          ['CD14', 'LYZ', 'AIF1', 'MS4A6A', 'HLA-DQA1', 'IL1B'],
        'Macrophages':      ['CD163', 'MRC1', 'MERTK', 'C1QA'],
        'Mast':             ['CPA3', 'TPSB2', 'MS4A2', 'KIT'],
        'NK (all)':         ['NCAM1', 'KLRD1', 'CD160'],
        'NK CD103+':        ['ITGAE', 'CCL5', 'IL16', 'LDB2'],
        'NK CD11c+':        ['ITGAX', 'LILRB1', 'KLRC1', 'GNLY', 'GZMB', 'PRF1'],
        'T cells':          ['CD3D', 'CD3G', 'CD3E', 'IL7R', 'THEMIS'],
        'CD4 T':            ['CD4', 'ADAM19', 'CDC14A', 'PDE3B', 'CD40LG', 'CD226'],
        'CD8 T':            ['CD8A', 'CD8B', 'DTHD1', 'TC2N', 'IFNG-AS1', 'GZMK'],
        'Treg':             ['FOXP3', 'CTLA4', 'TSHZ2', 'TBC1D4'],
        'ILC3':             ['TRDC', 'RORC', 'IL1R1', 'IL23R', 'ABCB5', 'PCDH9', 'GATA3'],
        'B cells':          ['MS4A1', 'CD79A'],
        'Cycling':          ['TOP2A', 'KIF15', 'KNL1', 'MKI67', 'BUB1', 'DEPDC1', 'KIF11'],
        'Apoptosis':        ['CASP3', 'CASP9', 'TP53', 'BAX'],
        'Hormone receptor': ['ESR1', 'PGR', 'AR'],
    },
}
marker_dict = marker_dicts[SPECIES]

# keep only genes present in adata; drop empty groups
all_markers = [g for v in marker_dict.values() for g in v]
missing = [g for g in all_markers if g not in adata.var_names]
print(f"{SPECIES} markers not in adata ({len(missing)}/{len(all_markers)}):", missing)
if len(missing) > len(all_markers) / 2:
    print(f"WARNING: most {SPECIES} markers are missing; check that SPECIES matches the data")

plot_dict = {k: [g for g in v if g in adata.var_names] for k, v in marker_dict.items()}
plot_dict = {k: v for k, v in plot_dict.items() if v}
n_genes = sum(len(v) for v in plot_dict.values())

for res in resolutions:
    leiden_key = f'leiden_scVI_{res}'
    if leiden_key not in adata.obs:
        print(f"Resolution {res} not found in adata.obs. Skipping...")
        continue

    n_clusters = adata.obs[leiden_key].nunique()

    with plt.style.context('dark_background'):    # white text, group labels and brackets
        sc.pl.dotplot(
            adata,
            var_names=plot_dict,                  # dict -> group labels on top
            groupby=leiden_key,
            layer='log1p_counts',
            use_raw=False,
            title=f"{SPECIES.capitalize()} Marker Genes by Clusters (scVI {res})",
            figsize=(0.22 * n_genes + 3, 0.35 * n_clusters + 2),
            show=False
        )
        fig = plt.gcf()
        fig.patch.set_facecolor('black')
        for ax_ in fig.get_axes():
            ax_.set_facecolor('black')
        plt.show()

# 8. Visualize selected clusters on a UMAP

In [ ]:
sc.pl.embedding(adata,
                basis=SCVI_UMAP_KEY,
                color=f"leiden_scVI_{CLEAN_RES}",
                groups=low,
                na_color='lightgray',
                legend_loc='on data',
                title=f"Leiden {CLEAN_RES}")

# 9. Visualize top 20 marker genes of the selected clusters

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

cl = f"leiden_scVI_{CLEAN_RES}"
key = f"rgg_low_{CLEAN_RES}"

sc.tl.rank_genes_groups(adata,
                        groupby=cl,
                        groups=low,
                        reference='rest',
                        method='wilcoxon',
                        layer='log1p_counts',
                        use_raw=False,
                        key_added=key)

top20 = pd.DataFrame(adata.uns[key]['names']).head(20)
top20 = top20[sorted(top20.columns, key=int)]
print(f"Leiden {CLEAN_RES}")
display(top20)

# 10. Check QC metrics of the selected clusters

In [ ]:
cl = f"leiden_scVI_{CLEAN_RES}"

qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt',
              'pct_counts_ribo', 'doubletdetection_doublet_score']

labels = adata.obs[cl].astype(str)
if low is not None:
    low_str = sorted({str(c) for c in low}, key=int)
else:
    low_str = sorted(labels.unique(), key=int)

grp = pd.Categorical(labels.where(labels.isin(low_str), 'other'),
                     categories=low_str + ['other'])

qc_table = adata.obs[qc_metrics].groupby(grp, observed=True).median().round(2)
qc_table.insert(0, 'n_cells', adata.obs.groupby(grp, observed=True).size())
qc_table.index.name = cl

print(qc_table.to_string())

# 11. Check the contribution of each sample to the selected cluster

In [ ]:
cl = f"leiden_scVI_{CLEAN_RES}"

obs = adata.obs
if low is not None:
    obs = obs[obs[cl].astype(str).isin([str(c) for c in low])]

ct = pd.crosstab(obs[cl].astype(str), obs['sample'],
                 margins=True, margins_name='Total')

# numeric cluster order, Total row last
order = sorted([i for i in ct.index if i != 'Total'], key=int) + ['Total']
ct = ct.loc[order]

print(ct.to_string())

# 12. Delete Bad Quality Clusters

In [ ]:
cl = f"leiden_scVI_{CLEAN_RES}"
rm = [str(c) for c in remove] if remove else []

mask = adata.obs[cl].astype(str).isin(rm)
print(f"Removing clusters {rm} from {cl}: {mask.sum()} cells")
print(adata.obs.loc[mask, cl].astype(str).value_counts().loc[sorted(rm, key=int)].to_string())

adata = adata[~mask].copy()
print(f"Remaining: {adata.n_obs} cells")

# 13. Annotate Cell types

In [ ]:
def annotate(adata, res, mapping, col):
    """Write labels to adata.obs[col] from {'label': ['0', '1'] or 'all'} at leiden_scVI_{res}."""
    cl = f"leiden_scVI_{res}"
    if cl not in adata.obs:
        raise KeyError(f"{cl} not found; add {res} to `resolutions` and rerun the clustering")
    labels = adata.obs[cl].astype(str)

    # single label for every cell
    if len(mapping) == 1 and list(mapping.values())[0] == 'all':
        label = list(mapping)[0]
        adata.obs[col] = pd.Categorical([label] * adata.n_obs, categories=[label])
        print(f"{col}: all {adata.n_obs} cells -> '{label}'")
        return

    # invert {'label': clusters} -> {'cluster': label}
    c2l = {}
    for label, clusters in mapping.items():
        for c in map(str, clusters):
            if c in c2l:
                raise ValueError(f"cluster {c} assigned to both '{c2l[c]}' and '{label}'")
            c2l[c] = label

    unmapped = sorted(set(labels) - set(c2l), key=int)
    if unmapped:
        print(f"{col}: clusters not in map -> 'unassigned': {unmapped}")

    adata.obs[col] = pd.Categorical(labels.map(c2l).fillna('unassigned'),
                                    categories=list(mapping) + ['unassigned'])

    print(pd.crosstab(labels, adata.obs[col]).loc[sorted(labels.unique(), key=int)].to_string())


annotate(adata, TYPE_RES, type_map, 'cell_type')
print(adata.obs['cell_type'].value_counts().to_string())

# 14. Annotate Cell Subtypes

In [ ]:
annotate(adata, SUBTYPE_RES, subtype_map, 'cell_subtype')
print(adata.obs['cell_subtype'].value_counts().to_string())

# sanity check: how subtypes nest within types
print(pd.crosstab(adata.obs['cell_type'], adata.obs['cell_subtype']).to_string())

# 15. Save the data and the Model

In [ ]:
# Save the updated AnnData object
adata.X = adata.layers['raw_counts'].copy()
adata.write(output_file, compression="gzip")
print(f"Updated AnnData object saved to {output_file}")

# Save the model
model.save(model_dir, overwrite=True)